In [ ]:
import copy
import math
import multiprocessing
import os
import random
import time
import warnings
from collections.abc import Callable
from functools import partial

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib_inline.backend_inline import set_matplotlib_formats
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torch.utils.flop_counter import FlopCounterMode

from dataset_loading import get_batch, prepare

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# mixed precision (float16) training on CUDA GPUs
USE_AMP = device.type == "cuda"

# channels-last memory layout speeds up convolutions on CUDA GPUs
MEMORY_FORMAT = (
    torch.channels_last if device.type == "cuda" else torch.contiguous_format
)

# pick the fastest convolution algorithms for our shapes
torch.backends.cudnn.benchmark = True

print(
    f"PyTorch {torch.__version__}, device: {device}"
)

## Training loop

In [ ]:
from train import evaluate, fit, generate

## Model: LSTM

In [ ]:
class LSTMLM(nn.Module):
    """LSTM language model. Like a plain RNN, but with a gated memory cell, so information survives longer."""

    def __init__(self, vocab_size: int, d_model: int, hidden: int, n_layers: int):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)  # token id -> vector
        # per token: forget / input / output gates decide what the cell state c_t keeps, adds and reveals as h_t
        self.rnn = nn.LSTM(d_model, hidden, n_layers, batch_first=True)
        self.proj = nn.Linear(hidden, d_model, bias=False)  # back to embedding size so the head can share weights
        self.head = nn.Linear(d_model, vocab_size, bias=False)  # vector -> score for every token
        self.head.weight = self.tok_emb.weight  # weight tying, same as in the other models
        nn.init.normal_(self.tok_emb.weight, std=0.02)  # default N(0, 1) would make the tied logits huge

    def forward(self, idx):  # idx: (B, T) token ids -> logits (B, T, vocab_size)
        out, _ = self.rnn(self.tok_emb(idx))  # out: (B, T, hidden), h after every token; start states are zeros
        return self.head(self.proj(out))

## Build the model

In [ ]:
# --- config: same data setup as the other models, hidden size chosen to give a similar parameter count ---
BLOCK_SIZE, BATCH_SIZE = 256, 64
D_MODEL, HIDDEN, N_LAYERS = 384, 880, 2

tok = prepare()
model = LSTMLM(tok.get_vocab_size(), D_MODEL, HIDDEN, N_LAYERS).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

model.train()

## Train

In [ ]:
history = fit(model, steps=5000, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE)

In [ ]:
plt.plot(history["step"], history["train_loss"], label="train (fineweb-edu)")
plt.plot(history["step"], history["val_loss"], label="val (fineweb-edu)")
plt.xlabel("step"), plt.ylabel("cross-entropy"), plt.legend();

## Test on Wikipedia

In [ ]:
test_loss = evaluate(model, "test", BLOCK_SIZE, BATCH_SIZE, n_batches=100)
print(f"wikipedia test loss {test_loss:.3f}, perplexity {math.exp(test_loss):.1f}")

## Inference: prompt completions

In [ ]:
prompts = [
    "The capital of France is",
    "In 1969, the first humans",
    "Photosynthesis is the process by which",
    "The most important thing in a good education is",
    "To make a good cup of coffee, you should",
    "The history of the Roman Empire began",
    "Scientists have recently discovered that",
    "The main difference between a virus and a bacterium is",
    "When learning a new language, it helps to",
    "The president announced yesterday that",
]
for p in prompts:
    print(f"PROMPT: {p}\nOUTPUT: {generate(model, tok, p, max_new_tokens=60, block_size=BLOCK_SIZE)}\n")

## Inference time

In [ ]:
# inference speed: generate a fixed number of tokens (stop_at_eot=False so the length is exact)
N_NEW = 200
generate(model, tok, prompts[0], 10, block_size=BLOCK_SIZE, stop_at_eot=False)  # warm-up
if device.type == "cuda":
    torch.cuda.synchronize()

start = time.time()
generate(model, tok, prompts[0], N_NEW, block_size=BLOCK_SIZE, stop_at_eot=False)
if device.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.time() - start

print(f"{N_NEW} tokens in {elapsed:.2f} s -> {N_NEW / elapsed:.1f} tokens/s")